> **Disclaimer.** This project is educational and is not investment advice. Backtest and optimizer results are historical and do not predict future results.

# IFR2 across tickers: which ones work, and does it beat the CDI?

The single-ticker notebook (`ifr2.ipynb`) asks which parameters work on one stock. This one asks which **tickers** work with a setup, and whether the result beats leaving the money in the CDI. The rules live in `docs/brief.md` section 7, the score in `docs/score.md` and the benchmark in `docs/cdi.md`. The notebook holds no trading logic.

**Before trusting any number here:**
- **Survivorship bias.** The list is today's most traded names. Names that were delisted or fell out of the index are not in it, which flatters a mean-reversion strategy. Liquid names reduce the problem but do not remove it.
- **Picking the best of many tickers is the same trap as picking the best of many parameters.** A ticker that looks great in-sample must show it out-of-sample too, and even then B3 stocks move together, so tickers are not independent samples.
- **Beating the CDI is a minimum, not proof.** The comparison ignores risk: read `excess` next to the drawdown and Calmar. Idle cash earns nothing in the strategy, and taxes and slippage are not modelled. Prices are raw (dividends and splits ignored).
- Daily candles, entry at the signal close, sizing by ATR with no stop loss: see `ifr2.ipynb` for what each of those means.

In [ ]:
import csv
import json
import urllib.request
from datetime import UTC, date, datetime, timedelta
from pathlib import Path

import pandas as pd
import yfinance as yf

from b3core.ifr2 import Ifr2Config
from b3core.optimizer import ParamRange, ParamSpace, Constraints, TrialCache, optimize
from b3core.strategy import Ifr2Strategy
from b3core.trialstore import SqliteTrialStore
from b3core.universe import rank_across_tickers, scan_universe

pd.options.display.float_format = "{:,.4f}".format
CACHE = Path("../samples/ifr2_universe")  # gitignored vendor data: delete the folder to refresh
CACHE.mkdir(parents=True, exist_ok=True)

## 1. Parameters

The 30 liquid names of the research prototype. Replace or extend the list as you like; each name needs at least three years of history to be used. `OOS_FRACTION` sets one shared out-of-sample date for every ticker: the date that leaves roughly the last 30% of the common history out. Everything is chosen on the in-sample part only.

In [ ]:
TICKERS = [
    "PETR4", "VALE3", "ITUB4", "BBDC4", "BBAS3", "ABEV3", "WEGE3", "RENT3", "LREN3", "MGLU3",
    "SUZB3", "GGBR4", "CSNA3", "USIM5", "JBSS3", "RADL3", "RAIL3", "EQTL3", "CMIG4", "ELET3",
    "TOTS3", "PRIO3", "CSAN3", "BBSE3", "SBSP3", "VIVT3", "B3SA3", "BRAP4", "GOAU4", "CYRE3",
]
START = date(2012, 1, 1)
END = date(2027, 1, 1)  # exclusive
OOS_FRACTION = 0.3
MIN_CANDLES_IS = 750  # about three years of daily candles

CONFIG = Ifr2Config.preset(
    rsi_entry=30, exit_mode="target", target_lookback=2, max_days=None,
    atr_period=5, atr_mult=2, max_risk_brl=2_000, capital_brl=100_000, lot_size=1,
)

## 2. Load candles

Daily candles from Yahoo Finance, raw prices, one CSV per ticker in the cache folder. Tickers that fail to download or have too little history are listed, never silently dropped.

In [ ]:
def load_candles(ticker: str) -> pd.DataFrame | None:
    path = CACHE / f"{ticker}.csv"
    if not path.exists():
        raw = yf.download(f"{ticker}.SA", start=START, end=END, auto_adjust=False, progress=False)
        if raw.empty:
            return None
        raw.columns = raw.columns.get_level_values(0)
        raw = raw.dropna(subset=["Open", "High", "Low", "Close"])
        raw[["Open", "High", "Low", "Close", "Volume"]].to_csv(path)
    raw = pd.read_csv(path, index_col=0, parse_dates=True)
    return pd.DataFrame(
        {
            "ts": raw.index.tz_localize("America/Sao_Paulo").tz_convert("UTC"),
            "open": raw["Open"].to_numpy(),
            "high": raw["High"].to_numpy(),
            "low": raw["Low"].to_numpy(),
            "close": raw["Close"].to_numpy(),
            "volume": raw["Volume"].to_numpy(dtype=float),
        }
    ).reset_index(drop=True)


loaded = {t: load_candles(t) for t in TICKERS}
missing = [t for t, c in loaded.items() if c is None]
short = [t for t, c in loaded.items() if c is not None and len(c) < MIN_CANDLES_IS]
data = {t: c for t, c in loaded.items() if c is not None and len(c) >= MIN_CANDLES_IS}
print(f"{len(data)} tickers loaded; no data: {missing or 'none'}; too short: {short or 'none'}")

## 3. One shared out-of-sample date

The union of all trading dates is cut at the point that leaves the last 30% out. Every ticker is split at that same date, so in-sample and out-of-sample mean the same calendar periods for everyone.

In [ ]:
local_dates = {
    t: pd.DatetimeIndex(c["ts"]).tz_convert("America/Sao_Paulo").tz_localize(None).normalize()
    for t, c in data.items()
}
all_dates = sorted(set().union(*local_dates.values()))
CUT = all_dates[int(len(all_dates) * (1 - OOS_FRACTION))]
in_sample = {t: c[(local_dates[t] < CUT)].reset_index(drop=True) for t, c in data.items()}
out_of_sample = {t: c[(local_dates[t] >= CUT)].reset_index(drop=True) for t, c in data.items()}
print(f"out-of-sample starts {CUT.date()}; in-sample {all_dates[0].date()} to {all_dates[all_dates.index(CUT) - 1].date()}")

## 4. The CDI

The benchmark is the daily CDI, compounded day by day over each ticker's own window and compared with the strategy's CAGR (`docs/cdi.md`). Only business days may be compounded, so the series is restricted to the trading days found in the candles.

Two sources, in this order:
1. The Central Bank's series SGS 12 (the CDI itself). Its API blocks some networks.
2. If that fails, the **Selic target rate** from Ipeadata (the government's economic data service), minus 0.10 points a year, which is where the CDI normally sits. This is an **approximation**: checked against the CDI of 2019 to 2023 it is within about 0.1 points a year. The cell prints which source it used.

To use the exact CDI, save series 12 as `samples/ifr2_universe/cdi.csv` with the columns `date` (dd/mm/yyyy) and `rate` (percent per business day).

In [ ]:
def fetch(url: str) -> str:
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=60) as response:
        return response.read().decode("utf-8")


def cdi_from_bcb() -> pd.Series:
    rows, begin = [], START
    while begin < END and begin < datetime.now(UTC).date():
        stop = min(begin + timedelta(days=3_000), END - timedelta(days=1))  # about 8 years
        url = (
            "https://api.bcb.gov.br/dados/serie/bcdata.sgs.12/dados"
            f"?formato=json&dataInicial={begin:%d/%m/%Y}&dataFinal={stop:%d/%m/%Y}"
        )
        rows += json.loads(fetch(url))  # raises when the API answers with an HTML error page
        begin = stop + timedelta(days=1)
    index = pd.to_datetime([r["data"] for r in rows], format="%d/%m/%Y")
    return pd.Series([float(r["valor"]) for r in rows], index=index).sort_index()


def cdi_from_selic_target() -> pd.Series:
    url = "http://www.ipeadata.gov.br/api/odata4/ValoresSerie(SERCODIGO='BM366_TJOVER366')"
    values = json.loads(fetch(url))["value"]
    annual = pd.Series(
        [v["VALVALOR"] for v in values], index=pd.to_datetime([v["VALDATA"][:10] for v in values])
    )
    daily = (1 + (annual - 0.10) / 100) ** (1 / 252) - 1  # percent a year to a business-day rate
    return (daily * 100).reindex(all_dates).dropna()  # trading days only


def load_cdi() -> tuple[pd.Series, str]:
    exact = CACHE / "cdi.csv"
    approx = CACHE / "cdi_from_selic_target.csv"
    if not exact.exists() and not approx.exists():
        try:
            cdi_from_bcb().rename_axis("date").rename("rate").to_csv(exact, date_format="%d/%m/%Y")
        except Exception as error:  # the API blocks some networks
            print(f"Central Bank API unavailable ({type(error).__name__}); using the Selic target")
            cdi_from_selic_target().rename_axis("date").rename("rate").to_csv(approx, date_format="%d/%m/%Y")
    path = exact if exact.exists() else approx
    frame = pd.read_csv(path)
    index = pd.to_datetime(frame["date"], format="%d/%m/%Y")
    series = pd.Series(frame["rate"].to_numpy(dtype=float), index=index).sort_index()
    label = "CDI (BCB series 12)" if path == exact else "Selic target minus 0.10 points (approximation)"
    return series, label


cdi, CDI_SOURCE = load_cdi()
print(f"source: {CDI_SOURCE}")
print(f"{len(cdi)} days, {cdi.index[0].date()} to {cdi.index[-1].date()}; compounded over the whole period: {(1 + cdi / 100).prod():.2f}x")

## 5. Scan: one setup on every ticker

The same `CONFIG` on each ticker, in-sample and out-of-sample side by side. `excess` is the strategy's CAGR minus the CDI's over the same window, and `beats_cdi` is true when it is positive. Read `max_equity_drawdown` and `calmar` next to it: a ticker can beat the CDI and still carry a drawdown that is not worth it. Ranked by the in-sample `score_v1`.

In [ ]:
scan_is = scan_universe(in_sample, CONFIG, cdi, min_candles=MIN_CANDLES_IS // 2)
scan_oos = scan_universe(out_of_sample, CONFIG, cdi, min_candles=120)
columns = ["n_trades", "result", "cagr", "excess", "beats_cdi", "max_equity_drawdown", "calmar", "score_v1"]
scan = scan_is[columns].add_prefix("is_").join(scan_oos[columns].add_prefix("oos_"), how="left")
scan = scan.sort_values("is_score_v1", ascending=False)
print("skipped in-sample:", scan_is.attrs["skipped"] or "none", "| out-of-sample:", scan_oos.attrs["skipped"] or "none")
scan

How many tickers beat the CDI, and how many of those still do out-of-sample? A ticker that wins in-sample and loses out-of-sample was probably fitted by luck of the period.

In [ ]:
both = scan.dropna(subset=["oos_beats_cdi"])
summary = pd.Series(
    {
        "tickers": len(scan),
        "beat the CDI in-sample": int(scan["is_beats_cdi"].sum()),
        "beat the CDI out-of-sample": int(both["oos_beats_cdi"].sum()),
        "beat it in both": int((both["is_beats_cdi"] & both["oos_beats_cdi"]).sum()),
        "median in-sample excess": scan["is_excess"].median(),
        "median out-of-sample excess": both["oos_excess"].median(),
    },
    name="summary",
)
summary.to_frame()

## 6. Optimization: the best setup for each ticker

The scan above used one default setup on every ticker, so it says little about what IFR2 can do when tuned. Here every ticker gets the same grid as `ifr2.ipynb`, searched on its **in-sample** part only and ranked by `score_v1` (at least 30 trades). The best setting per ticker is then run on that ticker's **out-of-sample** part.

This is the most overfitted view in the notebook: each ticker picks its own best of hundreds of combinations, so its in-sample number is flattered by luck. The honest number is the out-of-sample one, and the next section asks whether one setting can work across tickers. Results are cached in a SQLite registry, so the run can be interrupted and resumed.

In [ ]:
SPACE = ParamSpace(
    {
        "rsi_entry": ParamRange(5, 40, 5),
        "exit_mode": ["target", "rsi", "target_or_rsi"],
        "target_lookback": [2, 3, 5, 7, 10],
        "rsi_exit": [60, 70, 80],
        "max_days": [None, 5, 10],
    },
    Ifr2Strategy(),
)
PARAMS = SPACE.names
MIN_TRADES = 30
WORKERS = 8
store = SqliteTrialStore(Path("../trials_ifr2_universe.sqlite"))  # git-ignored
cache = TrialCache(store)

trials_is = {}
for ticker, candles in in_sample.items():
    found = optimize(
        candles, CONFIG, SPACE, objective="score_v1", constraints=Constraints(min_trades=MIN_TRADES),
        oos_fraction=0.0, workers=WORKERS, cache=cache, run_label=f"universe {ticker} in-sample",
    )
    trials_is[ticker] = found.trials_table()
print(f"{len(trials_is)} tickers x {SPACE.restricted(CONFIG).size} combinations; backtests computed {cache.misses}, from the registry {cache.hits}")

In [ ]:
def config_from(row: pd.Series) -> Ifr2Config:
    """The grid parameters of a table row, applied on top of CONFIG."""
    values = {name: (None if pd.isna(row[name]) else row[name]) for name in PARAMS}
    return SPACE.build_config(CONFIG, values)


def run_setups(setups: dict[str, Ifr2Config]) -> pd.DataFrame:
    """Each ticker with its own setup: in-sample and out-of-sample side by side."""
    cols = ["n_trades", "result", "cagr", "excess", "beats_cdi", "max_equity_drawdown", "calmar", "score_v1"]
    rows = []
    for ticker, config in setups.items():
        a = scan_universe({ticker: in_sample[ticker]}, config, cdi, min_candles=1)[cols].add_prefix("is_")
        b = scan_universe({ticker: out_of_sample[ticker]}, config, cdi, min_candles=1)[cols].add_prefix("oos_")
        rows.append(a.join(b))
    return pd.concat(rows)


best_rows = {t: tbl[tbl["feasible"]].sort_values("objective", ascending=False).iloc[0]
             for t, tbl in trials_is.items() if tbl["feasible"].any()}
per_ticker = run_setups({t: config_from(r) for t, r in best_rows.items()})
per_ticker = pd.DataFrame({name: [best_rows[t][name] for t in per_ticker.index] for name in PARAMS},
                          index=per_ticker.index).join(per_ticker).sort_values("is_score_v1", ascending=False)
per_ticker[PARAMS + ["is_cagr", "is_excess", "is_score_v1", "oos_cagr", "oos_excess", "oos_beats_cdi", "oos_max_equity_drawdown", "oos_score_v1"]]

Compare the three views on the **out-of-sample** period, where nothing was chosen on the data: the default setup from section 5, each ticker's own best setting, and (next section) one setting shared by all tickers. If tuning per ticker does not beat the default out-of-sample, the in-sample gain was fitting.

## 7. One setting across tickers

A setting that works on many tickers is more likely to be an edge than one that works on a single ticker. Each combination is ranked by its **median** score across the tickers (one lucky ticker cannot carry it), with the share of tickers where it made money and the worst ticker. A combination must pass the 30-trade gate on at least 20 tickers to be ranked. Only the in-sample part is used.

In [ ]:
ranked = rank_across_tickers(trials_is, PARAMS, min_trades=MIN_TRADES, min_tickers=20)
ranked.head(10)

The top three shared settings, run on every ticker, in-sample and out-of-sample. `beat` counts the tickers that beat the CDI.

In [ ]:
def summarize(label: str, table: pd.DataFrame) -> dict:
    return {
        "setup": label,
        "tickers": len(table),
        "IS median cagr": table["is_cagr"].median(),
        "OOS median cagr": table["oos_cagr"].median(),
        "OOS median excess": table["oos_excess"].median(),
        "OOS beat CDI": int(table["oos_beats_cdi"].sum()),
        "OOS median equity dd": table["oos_max_equity_drawdown"].median(),
        "OOS median score": table["oos_score_v1"].median(),
    }


rows = [summarize("default (section 5)", scan),
        summarize("each ticker's own best", per_ticker)]
shared = {}
for rank in range(3):
    config = config_from(ranked.iloc[rank])
    table = run_setups({t: config for t in in_sample})
    shared[rank] = table
    label = "shared #%d: " % (rank + 1) + ", ".join(f"{n}={ranked.iloc[rank][n]}" for n in PARAMS)
    rows.append(summarize(label, table))
pd.DataFrame(rows).set_index("setup")

**How to read it.** The CDI comparison and the drawdown say whether a setup is worth running; the gap between the in-sample and out-of-sample columns says how much of it was fitting. If every setup loses to the CDI out-of-sample, the answer is not "tune harder": with one position at a time most capital sits idle, which is the portfolio question for the next section.